# 04 · Rastreamento dos avisos introduzidos e removidos

**Objetivo.** Identificar, nos commits posteriores à adoção, os avisos do Ruff introduzidos por agentes e por uma amostra de commits humanos, e verificar se e quando cada aviso deixou de existir.

- **Lê:** `../out/piloto.csv`, `../out/<projeto>/commits_atribuidos.csv` e os repositórios clonados em `../repos/<projeto>/` (via `git`)
- **Grava:** `../out/<projeto>/alteracoes.csv` e `../out/<projeto>/avisos_introduzidos.csv` (um par por projeto), além de `../out/alteracoes.csv` e `../out/avisos_introduzidos.csv` (todos os projetos juntos)

Nos caminhos, `<projeto>` é o nome do repositório com `/` trocado por `__`.

**O que esta célula faz.** Define o rastreamento por projeto: lista os arquivos `.py` alterados em cada commit do first-parent (o histórico do branch principal, que segue só o primeiro pai de cada merge) depois da adoção e compara os avisos do Ruff (linter de Python) no blob (objeto do Git com o conteúdo de uma versão do arquivo) antes e depois de cada alteração; um aviso é novo quando aumenta a contagem do par (regra, texto da linha). Como linha de base, sorteia com semente fixa (42) commits sem agente em número próximo ao de commits de agente (no mínimo 10) e segue cada arquivo alterado pelos commits seguintes, inclusive renomeações, para registrar se o aviso foi corrigido ou o arquivo foi apagado e se o commit que o removeu é de agente. Para cada projeto grava `alteracoes.csv` (uma linha por arquivo alterado, com SLOC, as linhas de código sem brancos nem comentários, e avisos antes e depois) e `avisos_introduzidos.csv` (uma linha por aviso novo).

In [1]:
import bisect, importlib.util, json, os, re, subprocess, sys, tempfile
from collections import Counter, defaultdict
import pandas as pd
from radon.raw import analyze

RUFF_SELECT = 'F,E,B,SIM,C90,PLE,PLW,PLR,PLC,ERA'
RUFF_IGNORE = 'E501,PLR2004,PLC0415'
EXCL = re.compile(r'(^|/)(\.venv|venv|env|site-packages|node_modules|vendor|third_party|build|dist|\.tox)/')
RUFF = [sys.executable, '-m', 'ruff'] if importlib.util.find_spec('ruff') else ['ruff']
SEMENTE = 42


def git(repo, *args, entrada=None):
    r = subprocess.run(['git', '-C', repo, *args], capture_output=True, input=entrada)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.decode('utf-8', 'replace')[:300])
    return r.stdout


def eh_py(p):
    return p.endswith('.py') and not EXCL.search(p)


def sloc_de(src):
    try:
        return analyze(src).sloc
    except Exception:
        return sum(1 for l in src.splitlines() if l.strip() and not l.strip().startswith('#'))


def alteracoes_por_commit(repo, desde):
    """Passo 1: para cada commit do first-parent depois de `desde`, os arquivos alterados (com blob antes/depois)."""
    bruto = git(repo, 'log', '--first-parent', '-M', '--raw', '--no-abbrev', '--format=%x1e%H',
                f'{desde}..HEAD' if desde else 'HEAD').decode('utf-8', 'replace')
    mud = {}
    for rec in bruto.split('\x1e')[1:]:
        linhas = rec.strip().splitlines()
        sha, lista = linhas[0].strip(), []
        for l in linhas[1:]:
            if not l.startswith(':'):
                continue
            meta, *caminhos = l.split('\t')
            f = meta.split()
            antes, depois, st = f[2], f[3], f[4]
            velho = caminhos[0]
            novo = caminhos[1] if len(caminhos) > 1 else caminhos[0]
            lista.append((st[0], velho, novo, None if set(antes) == {'0'} else antes,
                          None if set(depois) == {'0'} else depois))
        mud[sha] = lista
    return mud


def ler_blobs(repo, blobs):
    blobs = sorted(blobs)
    saida = git(repo, 'cat-file', '--batch', entrada=('\n'.join(blobs) + '\n').encode())
    conteudo, i = {}, 0
    while i < len(saida):
        fim = saida.index(b'\n', i)
        sha, _, tam = saida[i:fim].decode().split()
        tam = int(tam)
        conteudo[sha] = saida[fim + 1: fim + 1 + tam].decode('utf-8', 'replace')
        i = fim + 1 + tam + 1
    return conteudo


def avisos_por_blob(conteudo):
    """Ruff em todos os blobs; cada aviso vira o par (regra, texto da linha)."""
    res = defaultdict(list)
    with tempfile.TemporaryDirectory() as tmp:
        for b, src in conteudo.items():
            with open(os.path.join(tmp, b + '.py'), 'w', encoding='utf-8') as f:
                f.write(src)
        r = subprocess.run([*RUFF, 'check', '--isolated', '--no-cache', '--exit-zero', '--output-format', 'json',
                            '--select', RUFF_SELECT, '--ignore', RUFF_IGNORE, tmp], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError('Ruff falhou: ' + r.stderr[:300])
        for a in json.loads(r.stdout or '[]'):
            if a['code'] is None:
                continue
            b = os.path.basename(a['filename'])[:-3]
            linhas = conteudo[b].splitlines()
            row = a['location']['row']
            res[b].append((a['code'], linhas[row - 1].strip() if 0 < row <= len(linhas) else ''))
    return res


def rastrear(nome, base='..'):
    chave = nome.replace('/', '__')
    repo, saida = f'{base}/repos/{chave}', f'{base}/out/{chave}'
    h = pd.read_csv(f'{saida}/commits_atribuidos.csv')
    h['data'] = pd.to_datetime(h.data, utc=True)
    h = h.sort_values('data').reset_index(drop=True)
    adocao = h[h.is_agente].data.min()
    antes_adocao = h[h.data < adocao]
    mud = alteracoes_por_commit(repo, antes_adocao.iloc[-1].sha if len(antes_adocao) else None)
    pos = {s: i for i, s in enumerate(h.sha)}
    data_de, ag_de, nome_ag = dict(zip(h.sha, h.data)), dict(zip(h.sha, h.is_agente)), dict(zip(h.sha, h.agente))

    # (commit, arquivo) alterados depois da adoção
    linhas = []
    for c in h[h.data >= adocao].itertuples():
        for st, velho, novo, antes, depois in mud.get(c.sha, []):
            if eh_py(novo) and st != 'D':
                linhas.append((c.sha, c.data, novo, bool(c.is_agente), c.agente, antes, depois))
    toque = pd.DataFrame(linhas, columns=['sha', 'data', 'caminho', 'is_agente', 'agente', 'antes', 'depois'])
    # 2. linha de base humana: mesmo número de commits, sorteados com semente fixa
    ag_shas = toque[toque.is_agente].sha.unique()
    hum = toque[~toque.is_agente]
    hum_shas = pd.Series(hum.sha.unique())
    if len(hum_shas):
        hum_shas = hum_shas.sample(n=min(len(hum_shas), max(len(ag_shas), 10)), random_state=SEMENTE)
    sel = pd.concat([toque[toque.is_agente], hum[hum.sha.isin(set(hum_shas))]]).drop_duplicates(['sha', 'caminho'])
    rastro = [(t.sha, t.caminho, t.antes if isinstance(t.antes, str) else None,
               t.depois if isinstance(t.depois, str) else None) for t in sel.itertuples()]

    # 4. seguir o arquivo nos commits seguintes (com renomeações)
    eventos = defaultdict(list)
    for sha, lista in mud.items():
        if sha in pos:
            for st, velho, novo, antes, depois in lista:
                eventos[velho].append((pos[sha], st, novo, depois, sha))
    for k in eventos:
        eventos[k].sort()

    def seguir(sha_intro, caminho):
        atual, i, saida = caminho, pos[sha_intro], []
        while True:
            ev = eventos.get(atual, [])
            j = bisect.bisect_right([e[0] for e in ev], i)
            mudou = False
            for pi, st, novo, depois, sha in ev[j:]:
                if st == 'D':
                    saida.append((sha, None)); return saida
                if st == 'R' and novo != atual:
                    saida.append((sha, depois)); atual, i, mudou = novo, pi, True
                    break
                saida.append((sha, depois)); i = pi
            if not mudou:
                return saida

    seg = {(s, p): seguir(s, p) for s, p, _, d in rastro if d}
    todos = {b for r in rastro for b in r[2:] if b} | {b for v in seg.values() for _, b in v if b}
    conteudo = ler_blobs(repo, todos)
    av = avisos_por_blob(conteudo)

    # 3 e 5. avisos novos e quem os removeu
    introduzidos, alter = [], []
    for sha, caminho, antes, depois in rastro:
        ca = Counter(av[depois]) if depois else Counter()
        cb = Counter(av[antes]) if antes else Counter()
        alter.append(dict(repo=nome, sha=sha, data=data_de[sha], autor_agente=bool(ag_de[sha]), agente=nome_ag[sha],
                          caminho=caminho, sloc_antes=sloc_de(conteudo[antes]) if antes else 0,
                          sloc_depois=sloc_de(conteudo[depois]) if depois else 0,
                          avisos_antes=sum(cb.values()), avisos_depois=sum(ca.values())))
        for chave_av, n in (ca - cb).items():
            for k in range(n):
                rem, como = None, None
                for s, b in seg.get((sha, caminho), []):
                    if b is None:
                        rem, como = s, 'arquivo removido'; break
                    if Counter(av[b])[chave_av] < ca[chave_av] - k:
                        rem, como = s, 'aviso corrigido'; break
                introduzidos.append(dict(
                    repo=nome, sha=sha, data=data_de[sha], autor_agente=bool(ag_de[sha]), agente=nome_ag[sha],
                    caminho=caminho, regra=chave_av[0], data_remocao=data_de.get(rem), como=como,
                    removido_por_agente=ag_de.get(rem) if rem else None))
    pd.DataFrame(alter).to_csv(f'{saida}/alteracoes.csv', index=False)
    pd.DataFrame(introduzidos).to_csv(f'{saida}/avisos_introduzidos.csv', index=False)
    print(f'{nome}: {len(alter)} alterações rastreadas ({len(ag_shas)} commits de agente), '
          f'{len(introduzidos)} avisos introduzidos')
    return pd.DataFrame(alter), pd.DataFrame(introduzidos)

**O que esta célula faz.** Executa o rastreamento nos cinco projetos, concatena os resultados em `../out/alteracoes.csv` e `../out/avisos_introduzidos.csv` e imprime, por projeto, as alterações e os avisos novos rastreados. A última linha cruza projeto e tipo de autor do commit (`False` = humano, `True` = agente) para contar os avisos introduzidos.

In [2]:
piloto = pd.read_csv('../out/piloto.csv')
res = [rastrear(p.full_name) for _, p in piloto.iterrows()]
alteracoes = pd.concat([a for a, _ in res], ignore_index=True)
avisos_introduzidos = pd.concat([b for _, b in res], ignore_index=True)
alteracoes.to_csv('../out/alteracoes.csv', index=False)
avisos_introduzidos.to_csv('../out/avisos_introduzidos.csv', index=False)
avisos_introduzidos.groupby(['repo', 'autor_agente']).size().unstack(fill_value=0)

numbagg/numbagg: 125 alterações rastreadas (25 commits de agente), 47 avisos introduzidos
econ-ark/HARK: 73 alterações rastreadas (14 commits de agente), 43 avisos introduzidos
NewFuture/DDNS: 495 alterações rastreadas (38 commits de agente), 271 avisos introduzidos
lidofinance/diffyscan: 45 alterações rastreadas (4 commits de agente), 25 avisos introduzidos
airbytehq/PyAirbyte: 871 alterações rastreadas (178 commits de agente), 173 avisos introduzidos


autor_agente,False,True
repo,,
NewFuture/DDNS,149,122
airbytehq/PyAirbyte,11,162
econ-ark/HARK,41,2
lidofinance/diffyscan,21,4
numbagg/numbagg,13,34


**O que isto mostra.** Foram rastreadas 125 alterações em numbagg, 73 em HARK, 495 em DDNS, 45 em diffyscan e 871 em PyAirbyte, incluindo 25, 14, 38, 4 e 178 commits de agente, respectivamente, com 47, 43, 271, 25 e 173 avisos introduzidos. A tabela separa esses avisos por autor (humano, agente): numbagg 13 e 34, HARK 41 e 2, DDNS 149 e 122, diffyscan 21 e 4, PyAirbyte 11 e 162. Verificação: em cada projeto a soma das duas colunas coincide com o total impresso acima (por exemplo, 13 + 34 = 47 em numbagg), e as colunas somam 235 (humano) e 324 (agente) no conjunto. São contagens brutas: os commits humanos são uma amostra sorteada de tamanho próximo ao dos commits de agente e os projetos diferem muito em volume (871 alterações em PyAirbyte contra 45 em diffyscan), então a comparação entre agentes e humanos exige a normalização feita no notebook 05.